# Forecasting climate data using Physics Informed Long Short-Term Memory networks (PI-LSTM)

This code implements the PINT LSTM ([Code](https://github.com/KV-Park/PINT/blob/main/PINT%5BTraining%5D.ipynb) | [Paper](https://arxiv.org/pdf/2502.04018)) to forecast on the Kaggle "Daily Climate time series data" dataset.

Dataset:
- Download dataset from [here](https://www.kaggle.com/datasets/sumanthvrao/daily-climate-time-series-data) and store in data/ folder in this repository

In [ ]:
import os
from datetime import datetime

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
from torch import nn
from matplotlib.dates import MonthLocator, DateFormatter
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler

from helper_functions import create_sequences, forecast_with_model, set_seed, plot_loss
from models import GRU, LSTM, RNN

In [ ]:
matplotlib.rcParams.update({'font.size': 16})

In [ ]:
# Key parameters:
SEED_VALUE = 2026

# Single-step prediction
PRED_LENGTH = 1  # Number of time steps to predict

# Model set up
input_size=1
num_layers=1
output_size = PRED_LENGTH
hidden_size=64
dropout=0
NUM_EPOCHS=50
LEARNING_RATE = 0.01

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
colours = {  # IBM colourblind palette
    "Ground truth": "#1589e8",
    "LSTM": "#dc267f",
    "PI-LSTM": "#dc267f",
    "Starting data": "#631ff3",
    "RNN": "#fe5100",
    "PI-RNN": "#fe5100",
    "GRU": "#ffb000",
    "PI-GRU": "#ffb000"
}

In [ ]:
set_seed(SEED_VALUE)

In [ ]:
# # Initialise models
# models = {
#     "RNN": RNN(input_size, hidden_size, num_layers, output_size),
#     "LSTM": LSTM(input_size, hidden_size, num_layers, output_size),
#     "GRU": GRU(input_size, hidden_size, num_layers, output_size)
# }

## Import data

In [ ]:
files = ["DailyDelhiClimateTrain.csv", "DailyDelhiClimateTest.csv"]

data = []
for file in files:
    df = pd.read_csv(os.path.join("data/", file))
    df["datetimes"] = [datetime.strptime(d,'%Y-%m-%d').date() for d in df["date"].values]
    if "Train" in file:
        # print("End of \"Train\" data before:\n", df.tail())  # Shows 2017-01-01 at end
        df = df[:-1]  # Remove last row of training data because of duplcate in test
        # print("End of \"Train\" data after dropping last row:\n", df.tail())  # Shows 2017-01-01 at end
    # else:
        # print("Start of \"Test\" data:\n", df.head())  # Shows 2017-01-01 at start
    data.append(df)

all_data_df = pd.concat(data, ignore_index=True)

## Pre-process data

In [ ]:
# split training:val:test as 70:15:15
train_df = all_data_df.loc[:int(len(all_data_df)*0.7)].copy()
val_df = all_data_df.loc[int(len(all_data_df)*0.7):int(len(all_data_df)*0.85)].copy()
test_df = all_data_df.loc[int(len(all_data_df)*0.85):].copy()

In [ ]:
# Plot temperature variation across training dataset
fig = plt.figure(figsize=(12,6))
plt.plot(train_df["datetimes"], train_df["meantemp"], label="Training", c="#1589e8")
plt.plot(val_df["datetimes"], val_df["meantemp"], label="Validation", c="#dc267f")
plt.plot(test_df["datetimes"], test_df["meantemp"], label="Test", c="#ffb000")
ax = plt.gca()
ax.xaxis.set_major_locator(MonthLocator(interval=6))
ax.xaxis.set_major_formatter(DateFormatter('%b %Y'))
plt.xticks(rotation=90, ha="right")
ax.grid(visible=True)
plt.xlabel("Date")
plt.ylabel("Mean temperature (°C)")
plt.legend(bbox_to_anchor=(1.0, 1.0))

In [ ]:
# Normalise inputs to be between 0 and 1
scaler = MinMaxScaler()

train_df["meantemp_norm"] = scaler.fit_transform(train_df[["meantemp"]])
val_df["meantemp_norm"] = scaler.fit_transform(val_df[["meantemp"]])
test_df["meantemp_norm"] = scaler.fit_transform(test_df[["meantemp"]])

# Convert to numpy array for sequence creation
meantemp_train = train_df["meantemp_norm"].values
meantemp_val = val_df["meantemp_norm"].values
meantemp_test = test_df["meantemp_norm"].values

## Train, evaluate for different sequence lengths

In [ ]:
trained_models = {}
for SEQ_LENGTH in range(5, 90, 5):
    models = {
        "RNN": RNN(input_size, hidden_size, num_layers, output_size),
        "LSTM": LSTM(input_size, hidden_size, num_layers, output_size),
        "GRU": GRU(input_size, hidden_size, num_layers, output_size)
    }
    # Create sequences
    X_train, y_train = create_sequences(meantemp_train, SEQ_LENGTH, PRED_LENGTH)
    X_val, y_val = create_sequences(meantemp_val, SEQ_LENGTH, PRED_LENGTH)
    X_test, y_test = create_sequences(meantemp_test, SEQ_LENGTH, PRED_LENGTH)

    # Convert arrays to PyTorch tensors
    X_train = torch.Tensor(X_train).unsqueeze(-1) # Shape: (batch, seq, input_size)
    y_train = torch.Tensor(y_train)
    X_val = torch.Tensor(X_val).unsqueeze(-1) # Shape: (batch, seq, input_size)
    y_val = torch.Tensor(y_val)
    X_test = torch.Tensor(X_test).unsqueeze(-1)
    y_test = torch.Tensor(y_test)
    print(f"Train:val:test ratio: {len(X_train)}:{len(X_val)}:{len(X_test)}")

    print(f"=== Starting training for SEQ LENGTH = {SEQ_LENGTH} ===")
    for name, model in models.items():
        print(f"Training Standard Model: {name}")
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

        losses = []
        val_losses = []
        for epoch in range(NUM_EPOCHS):
            model.train()
            output = model(X_train)
            loss = criterion(output, y_train.reshape(-1,1))  # Compute loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            losses.append(loss.item())

            if (epoch + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{NUM_EPOCHS}], Loss: {loss.item():.4f}")

            # Validation
            model.eval()
            val_preds = model(X_val)
            val_loss = criterion(val_preds, y_val.reshape(-1,1))
            val_losses.append(val_loss.item())

        # plot_loss(losses, val_losses, NUM_EPOCHS)

        # Store models:
        trained_models[SEQ_LENGTH] = models

    print("Test set evaluation")
    y_preds = {}
    y_preds_inv = {}  # Inverse normalisation back to true temperature
    for name, model in models.items():
        model.eval()
        with torch.no_grad():
            preds = model(X_test).squeeze().numpy()
            y_preds[name] = preds # Predict on test set
            y_preds_inv[name] = np.array([scaler.inverse_transform(pred.reshape(-1, 1)) for pred in preds])
    
    for name in models.keys():
        # Calculate RMSE
        rmse = np.sqrt(mean_squared_error(y_test, y_preds[name]))

        # Calculate Pearson correlation
        correlation, _ = pearsonr(y_preds[name], y_test.squeeze())

        print(f"Metrics for model - {name}:")
        print(f"RMSE:\t\t{rmse}")
        print(f"P-CORR:\t\t{correlation}")
    
    # Plot
    y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1))
    x_start = scaler.inverse_transform(X_test[0])


    # Plot actual vs predicted 
    months = MonthLocator()
    plt.figure(figsize=(12, 5))
    plt.plot(test_df["datetimes"].values[:SEQ_LENGTH], x_start, label='First sequence', c=colours['Starting data'])
    plt.plot(test_df["datetimes"].values[SEQ_LENGTH:], y_test_inv, label='Ground truth', c=colours['Ground truth'])

    for name in models.keys():
        plt.plot(test_df["datetimes"].values[SEQ_LENGTH:], y_preds_inv[name].reshape(-1,1), label=name, c=colours[name])
    ax = plt.gca()
    ax.xaxis.set_major_locator(months)
    ax.xaxis.set_major_formatter(DateFormatter('%b %Y'))
    plt.xticks(rotation=90, ha="right")
    plt.grid(visible=True)
    plt.xlabel("Month")
    plt.ylabel("Mean Temperature (°C)")
    plt.legend()#bbox_to_anchor=(1,1))
    plt.show()

    # Plot correlation
    plt.figure(figsize=(6, 6))
    for name in models.keys():
        plt.scatter(y_test.reshape(-1,1), y_preds[name].reshape(-1,1), alpha=0.5, label=name, c=colours[name])
    plt.plot([y_test.min(), y_test.max()], 
            [y_test.min(), y_test.max()], 
            '--', lw=2, c=colours['Ground truth'])
    plt.xlabel('Actual Temperature')
    plt.ylabel('Predicted Temperature')
    plt.grid(True)
    plt.legend()
    plt.show()

    # Autoregressive
    print("Autoregressive forcasting")
    y_forecast = {}
    start_data = X_test[0:1]  # Starting data: first sequence of X_test
    desired_length = len(test_df)-SEQ_LENGTH
    for name, model in models.items():
        print(f"Forcasting for model: {name}")
        forecast = forecast_with_model(model, start_data, desired_length, PRED_LENGTH)
        y_forecast[name] = scaler.inverse_transform(forecast.reshape(-1,1))
    
    true_temps = test_df["meantemp"].values[SEQ_LENGTH:]
    forecast_dates = test_df["datetimes"].values[SEQ_LENGTH:]

    for name in models.keys():
        # Calculate RMSE
        rmse = np.sqrt(np.mean((y_forecast[name].squeeze() - true_temps) ** 2))

        # Calculate Pearson correlation
        correlation, _ = pearsonr(y_forecast[name].squeeze(), true_temps)

        print(f"Metrics for model - {name}:")
        print(f"RMSE:\t\t{rmse}")
        print(f"P-CORR:\t\t{correlation}")
    
    # Plot actual vs predicted
    plt.figure(figsize=(10, 4))

    # Starting data: first sequence of X_test
    start_data_inv = scaler.inverse_transform(start_data.reshape(-1,1)).squeeze()
    start_data_dates = test_df["datetimes"].values[:SEQ_LENGTH]
    plt.plot(start_data_dates, start_data_inv, marker='.', label='Starting data', c=colours['Starting data'])

    # Ground truth: test data across region predicted for
    plt.plot(forecast_dates, true_temps, label='Ground truth', marker='.', c=colours['Ground truth'])

    # Predictions
    for name in models.keys():
        ls = '--' if "PI" in name else '-'
        m = 'x' if "PI" in name else '.'
        plt.plot(forecast_dates, y_forecast[name], label=name, marker=m, c=colours[name], ls=ls)
    # plt.title("Autoregressive Forecasted Data")
    ax = plt.gca()
    ax.xaxis.set_major_locator(MonthLocator(interval=1))
    ax.xaxis.set_major_formatter(DateFormatter('%b %Y'))
    plt.xticks(rotation=90)
    plt.xlabel("Month")
    plt.ylabel("Mean Temperature (°C)")
    plt.grid(True)
    plt.legend(bbox_to_anchor=(1,1))
    plt.show()